In [6]:
!pip install -q transformers==4.40.2
!pip install -q datasets==2.18.0
!pip install -q accelerate==0.30.1
!pip install -q evaluate==0.4.2
!pip install -q jiwer==4.0.0

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
import torch
import re
import evaluate
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [33]:
fleurs = load_dataset("google/fleurs", "hi_in", split="test")
print("Total test samples:", len(fleurs))

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for google/fleurs contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/google/fleurs
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Total test samples: 418


In [34]:
print(fleurs.column_names)
print(fleurs[0])

['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']
{'id': 1766, 'num_samples': 145920, 'path': '/root/.cache/huggingface/datasets/downloads/extracted/45a0eac0974e44be76c31b251083e7c362e6d49ae6dde22b7729e0a8eab15dc6/10011266027513218401.wav', 'audio': {'path': 'test/10011266027513218401.wav', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00048101,
       -0.00045913, -0.00040722]), 'sampling_rate': 16000}, 'transcription': 'कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है', 'raw_transcription': 'कुछ अणुओं में अस्थिर केंद्रक होता है, जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है,', 'gender': 0, 'lang_id': 32, 'language': 'Hindi', 'lang_group_id': 4}


In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.to(device)
model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hindi",
    task="transcribe"
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [36]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\u0900-\u097F\s]", "", text)  # keep only Hindi characters
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="hi",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.to(device)
model.eval()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

In [43]:
def normalize_text(text):
    text = text.lower()
    return " ".join(text.split())

In [31]:
wer_metric = evaluate.load("wer")

predictions = []
references = []

for sample in tqdm(fleurs):

    # Extract audio features
    input_features = processor(
        sample["audio"]["array"],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    # Generate prediction
    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    # Store normalized texts
    predictions.append(normalize_text(transcription))
    references.append(normalize_text(sample["transcription"]))

# Compute WER
baseline_wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

print("Final Pretrained Whisper-small WER:", baseline_wer)

100%|██████████| 418/418 [16:40<00:00,  2.39s/it]

Final Pretrained Whisper-small WER: 0.8462514642717689


In [15]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/ASR/final-20260220T123411Z-1-001.zip"
EXTRACT_PATH = "/content/drive/MyDrive/ASR"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Unzip complete.")

Unzip complete.


In [16]:
import os
print(os.listdir("/content/drive/MyDrive/ASR"))

['Untitled0.ipynb', 'final-20260220T123411Z-1-001.zip', 'final']


In [37]:
from transformers import WhisperForConditionalGeneration

pretrained_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
pretrained_model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")

wer_metric = evaluate.load("wer")
baseline_predictions = []
baseline_references = []

for sample in tqdm(fleurs):
    input_features = processor(
        sample["audio"]["array"],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = pretrained_model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    baseline_predictions.append(normalize_text(transcription))
    baseline_references.append(normalize_text(sample["transcription"]))

baseline_wer = wer_metric.compute(predictions=baseline_predictions, references=baseline_references)

print("Pretrained Whisper-small WER:", baseline_wer)

100%|██████████| 418/418 [16:39<00:00,  2.39s/it]

Pretrained Whisper-small WER: 0.8290783637259734


In [44]:
print("\n-------- Final Results --------")
print(f"Pretrained Whisper-small WER : {baseline_wer:.4f}")
print(f"Fine-Tuned Whisper-small WER : {finetuned_wer:.4f}")
print(f"Improvement                  : {((baseline_wer - finetuned_wer) / baseline_wer * 100):.1f}% WER reduction")


-------- Final Results --------
Pretrained Whisper-small WER : 0.8291
Fine-Tuned Whisper-small WER : 0.3184
Improvement                  : 61.6% WER reduction


In [46]:
import pandas as pd

# Create results dataframe
results_df = pd.DataFrame({
    "Model": [
        "Whisper Small (Pretrained)",
        "FT Whisper Small (Fine-Tuned)"
    ],
    "WER": [
        baseline_wer,
        finetuned_wer
    ]
})

# Save CSV
results_df.to_csv("wer_results.csv", index=False)

print("wer_results.csv saved successfully ✅")
results_df

wer_results.csv saved successfully ✅


,Model,WER
0,Whisper Small (Pretrained),0.829078
1,FT Whisper Small (Fine-Tuned),0.318355
